# 02a Byte BPE Generation

## Purpose

This notebook trains the byte-level BPE tokenizer for the rebuilt workflow.

## Input

- `MyDrive/ProjectRoot2/data/splits/train.txt`

## Outputs

- `MyDrive/ProjectRoot2/tokenizers/byte_bpe/<setting_label>/vocab.json`
- `MyDrive/ProjectRoot2/tokenizers/byte_bpe/<setting_label>/merges.txt`
- Hugging Face tokenizer files saved in the same folder
- `tokenizer_config_summary.json`

## Notes to myself

This is the plain baseline tokenizer. I want this one in place first because it gives me the most neutral comparison point for the later tokenizers.

## Setup note

Same pattern again.

- code and notebooks stay in GitHub
- tokenizer artifacts stay in Drive
- Colab pulls the repo at the start
- the final cell syncs the notebook back to GitHub

In [ ]:
# ==============================================================================
# 0. SET UP THE COLAB ENVIRONMENT
# ==============================================================================
import os
import sys

from google.colab import drive, userdata

drive.mount('/content/drive')

GITHUB_USER = 'hb791-dev'
REPO_NAME = 'glycan-roberta'
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
REPO_DIR = f'/content/{REPO_NAME}'

if not os.path.exists(REPO_DIR):
    print('Cloning repository...')
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    print('Repository already exists. Pulling latest changes...')

%cd {REPO_DIR}

!git config --global user.email "hb791-dev@users.noreply.github.com"
!git config --global user.name "hb791-dev"
!git config --global pull.rebase false
!git pull {REPO_URL} main --no-edit -q

if REPO_DIR not in sys.path:
    sys.path.append(REPO_DIR)

print('Colab environment ready.')
print(f'Repo directory: {REPO_DIR}')

## Path and setting setup

Here I define the training file and the tokenizer setting label. I want the folder name to come directly from the tokenizer hyperparameters so it is obvious what was trained.

In [ ]:
# ==============================================================================
# 1. DEFINE THE TRAINING PATHS AND TOKENIZER SETTINGS
# ==============================================================================
PROJECT_ROOT = '/content/drive/MyDrive/ProjectRoot2'
TRAIN_DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'splits', 'train.txt')

VOCAB_SIZE = 300
MIN_FREQUENCY = 2
SETTING_LABEL = f'v{VOCAB_SIZE}_m{MIN_FREQUENCY}'

TOKENIZER_OUT_DIR = os.path.join(PROJECT_ROOT, 'tokenizers', 'byte_bpe', SETTING_LABEL)
os.makedirs(TOKENIZER_OUT_DIR, exist_ok=True)

print('Training data path:')
print(TRAIN_DATA_PATH)
print('\nTokenizer output directory:')
print(TOKENIZER_OUT_DIR)
print('\nSetting label:')
print(SETTING_LABEL)

if not os.path.exists(TRAIN_DATA_PATH):
    raise FileNotFoundError(f'Training split not found: {TRAIN_DATA_PATH}')

## Train the tokenizer

This is the actual byte-BPE training step. I am fixing the vocabulary size and minimum token frequency here so the setting label and output folder match the training choices.

In [ ]:
# ==============================================================================
# 2. TRAIN AND SAVE THE BYTE-LEVEL BPE TOKENIZER
# ==============================================================================
import json

from tokenizers import ByteLevelBPETokenizer
from transformers import PreTrainedTokenizerFast

print(f'Training byte-level BPE tokenizer (vocab={VOCAB_SIZE}, min_frequency={MIN_FREQUENCY})...')

tokenizer = ByteLevelBPETokenizer()

tokenizer.train(
    files=[TRAIN_DATA_PATH],
    vocab_size=VOCAB_SIZE,
    min_frequency=MIN_FREQUENCY,
    special_tokens=['<s>', '<pad>', '</s>', '<unk>', '<mask>']
)

tokenizer.save_model(TOKENIZER_OUT_DIR)

hf_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer._tokenizer,
    bos_token='<s>',
    eos_token='</s>',
    unk_token='<unk>',
    pad_token='<pad>',
    mask_token='<mask>'
)

hf_tokenizer.save_pretrained(TOKENIZER_OUT_DIR)

tokenizer_summary = {
    'tokenizer_family': 'byte_bpe',
    'setting_label': SETTING_LABEL,
    'vocab_size': VOCAB_SIZE,
    'min_frequency': MIN_FREQUENCY,
    'train_data_path': TRAIN_DATA_PATH,
    'tokenizer_output_dir': TOKENIZER_OUT_DIR,
    'saved_files': sorted(os.listdir(TOKENIZER_OUT_DIR)),
}

summary_json_path = os.path.join(TOKENIZER_OUT_DIR, 'tokenizer_config_summary.json')
with open(summary_json_path, 'w', encoding='utf-8') as file:
    json.dump(tokenizer_summary, file, indent=2)

print('Tokenizer training complete.')
print(f'Tokenizer saved to: {TOKENIZER_OUT_DIR}')

## Quick sanity check

I do not want to do deep analysis here. I just want to confirm the tokenizer loads, has the expected vocabulary size, and can tokenize a couple of example glycans from the training split.

In [ ]:
# ==============================================================================
# 3. LOAD THE SAVED TOKENIZER AND INSPECT SAMPLE OUTPUT
# ==============================================================================
import pandas as pd

with open(TRAIN_DATA_PATH, 'r', encoding='utf-8') as file:
    train_sequences = [line.strip() for line in file if line.strip()]

loaded_tokenizer = PreTrainedTokenizerFast.from_pretrained(TOKENIZER_OUT_DIR)

sample_sequences = train_sequences[:3]
inspection_rows = []

for sample_index, sequence in enumerate(sample_sequences, start=1):
    token_ids = loaded_tokenizer.encode(sequence, add_special_tokens=False)
    tokens = loaded_tokenizer.convert_ids_to_tokens(token_ids)

    inspection_rows.append(
        {
            'sample_index': sample_index,
            'sequence': sequence,
            'num_tokens': len(tokens),
            'tokens': ' | '.join(tokens[:30]),
        }
    )

inspection_df = pd.DataFrame(inspection_rows)
display(inspection_df)

print(f'Loaded vocabulary size: {len(loaded_tokenizer)}')
print(f'Mask token: {loaded_tokenizer.mask_token}')
print(f'Pad token: {loaded_tokenizer.pad_token}')

## Save a small inspection table

This just gives me a lightweight record of the first sanity check without opening the notebook again later.

In [ ]:
# ==============================================================================
# 4. SAVE THE INSPECTION OUTPUT
# ==============================================================================
inspection_path = os.path.join(TOKENIZER_OUT_DIR, 'inspection_preview.csv')
inspection_df.to_csv(inspection_path, index=False)

print(f'Inspection preview saved to: {inspection_path}')

## GitHub sync note

Same pattern as the earlier notebooks. The tokenizer artifacts stay in Drive. The notebook stays versioned in GitHub.

In [ ]:
# ==============================================================================
# SAVE THE NOTEBOOK BACK TO GITHUB
# ==============================================================================
import json

REPO_NOTEBOOK_PATH = os.path.join(REPO_DIR, 'notebooks/02_tokenizer_generation/02a_byte_bpe_gen.ipynb')
NOTEBOOK_FILENAME = os.path.basename(REPO_NOTEBOOK_PATH)
DRIVE_NOTEBOOK_CANDIDATES = [
    f'/content/drive/MyDrive/Colab Notebooks/{NOTEBOOK_FILENAME}',
    f'/content/drive/MyDrive/{NOTEBOOK_FILENAME}',
]

source_notebook_path = None
for candidate in DRIVE_NOTEBOOK_CANDIDATES:
    if os.path.exists(candidate):
        source_notebook_path = candidate
        break

if source_notebook_path is not None:
    !cp "{source_notebook_path}" "{REPO_NOTEBOOK_PATH}"

    # Strip widget metadata if Colab adds it so GitHub rendering stays cleaner.
    try:
        with open(REPO_NOTEBOOK_PATH, 'r', encoding='utf-8') as file:
            notebook_json = json.load(file)

        if 'widgets' in notebook_json.get('metadata', {}):
            del notebook_json['metadata']['widgets']

        with open(REPO_NOTEBOOK_PATH, 'w', encoding='utf-8') as file:
            json.dump(notebook_json, file, indent=1)
    except Exception as exc:
        print(f'Notebook metadata cleanup skipped: {exc}')

    %cd {REPO_DIR}
    !git add notebooks/02_tokenizer_generation/02a_byte_bpe_gen.ipynb
    !git commit -m "Update 02a_byte_bpe_gen" || echo "No new changes to commit."
    !git pull {REPO_URL} main --no-edit -q
    !git push {REPO_URL} main -q

    print('Notebook synced to GitHub.')
    print(f'Source notebook path: {source_notebook_path}')
else:
    print('No Drive-backed notebook file was found for this session.')
    print('If you opened this notebook directly from GitHub, Colab is editing a browser copy, not a runtime file.')
    print('For GitHub-opened notebooks, use File -> Save a copy in GitHub.')
    print('If you want this cell to auto-sync the notebook, first save or copy the notebook into Drive and then rerun this cell.')
